# Notebook 04 — Insights & Visualisations

**Goal:** Produce all interactive Plotly charts and derive actionable personalization recommendations for each listener segment.

**Research question:** *What user segments exist based on listening diversity patterns, and how can streaming platforms use these insights to optimise personalization strategies for different listener types?*

**Inputs** (from `data/processed/`):
- `user_features.parquet`
- `cluster_labels.parquet`
- `scrobbles_updated.parquet`
- `artist_genres.parquet`
- `profiles.parquet`

**Outputs** (in `outputs/figures/`):
- `umap_clusters.html` — 2D UMAP scatter
- `cluster_heatmap.html` — feature profile heatmap
- `radar_chart.html` — radar comparison of segments
- `feature_importance.html` — discriminating features
- `genre_distribution.html` — genre breakdown per segment
- `temporal_ratios.html` — morning/evening/weekend ratio breakdown per segment
- `temporal_hour_distribution.html` — normalised hour-of-day listening curves per segment
- `temporal_heatmap_cluster_N.html` — hour × day-of-week activity heatmap per segment

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO)

import pandas as pd
import numpy as np
import plotly.io as pio

pio.renderers.default = 'notebook'
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
labels_df      = pd.read_parquet('../data/processed/cluster_labels.parquet')
scrobbles      = pd.read_parquet('../data/processed/scrobbles_updated.parquet')
profiles       = pd.read_parquet('../data/processed/profiles.parquet')

# artist_genres is only needed for the stacked genre chart (Section 5a)
# All other sections including temporal patterns run without it
_ag_path = Path('../data/processed/artist_genres.parquet')
artist_genres = pd.read_parquet(_ag_path) if _ag_path.exists() else None
if artist_genres is None:
    print('WARNING: artist_genres.parquet not found — Section 5a (genre distribution) will be skipped.')

labels = labels_df['cluster'].values
cluster_names = dict(zip(labels_df['cluster'], labels_df['cluster_name']))
userids = labels_df['userid'].tolist()

print(f'Users: {len(userids)} | Segments: {labels_df["cluster"].nunique()}')
labels_df['cluster_name'].value_counts()

In [ ]:
# Map auto-generated cluster labels to human-readable segment names
# Update this dict if you re-run clustering and the auto-labels change
_name_map = {
    "Slow-Discovery / Low-Replay":    "The Casual Listener",
    "Fast-Discovery / Low-Replay":    "The Discoverer",
    "Slow-Discovery / Short-Sessions": "The Loyalist",
    "High-Replay / Slow-Discovery":   "The Focused Listener",
    "Long-Sessions / Weekday-Heavy":  "The Binge Listener",
}
cluster_names = {cid: _name_map.get(name, name) for cid, name in cluster_names.items()}
print('Segment names:', list(cluster_names.values()))

## 1. UMAP Cluster Scatter

In [ ]:
from src.visualization.plots import plot_umap_clusters, save_figure

umap_coords = labels_df[['umap_x', 'umap_y']].values

fig_umap = plot_umap_clusters(
    umap_coords=umap_coords,
    labels=labels,
    cluster_names=cluster_names,
    userids=userids,
)
save_figure(fig_umap, '../outputs/figures/umap_clusters')
fig_umap.show()

## 2. Cluster Feature Heatmap

Z-scored means — red = above average, blue = below average.

In [ ]:
from src.visualization.plots import plot_cluster_heatmap
from src.clustering.evaluation import summarise_clusters

# Focus on the most interpretable features for the heatmap
key_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists', 'artist_concentration_20',
    'track_replay_rate', 'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'avg_session_length_min', 'weekend_ratio',
    'temporal_hour_entropy', 'morning_ratio', 'evening_ratio',
    'mean_energy', 'mean_valence', 'mean_danceability', 'mean_acousticness',
]
key_features = [f for f in key_features if f in feature_matrix.columns]

fig_heatmap = plot_cluster_heatmap(
    feature_matrix=feature_matrix,
    labels=labels,
    features=key_features,
    cluster_names=cluster_names,
)
save_figure(fig_heatmap, '../outputs/figures/cluster_heatmap')
fig_heatmap.show()

## 3. Radar Chart — Segment Profiles

In [ ]:
from src.visualization.plots import plot_cluster_radar

cluster_summary = summarise_clusters(feature_matrix, labels)

radar_features = [
    'artist_entropy', 'genre_entropy', 'novelty_ratio',
    'track_replay_rate', 'avg_tracks_per_session', 'temporal_hour_entropy',
]
radar_features = [f for f in radar_features if f in feature_matrix.columns]

fig_radar = plot_cluster_radar(
    cluster_summary=cluster_summary,
    features=radar_features,
    cluster_names=cluster_names,
)
save_figure(fig_radar, '../outputs/figures/radar_chart')
fig_radar.show()

## 4. Feature Importance

In [ ]:
from src.clustering.evaluation import feature_importance
from src.visualization.plots import plot_feature_importance

imp_df = feature_importance(feature_matrix, labels)

fig_imp = plot_feature_importance(imp_df, top_n=20)
save_figure(fig_imp, '../outputs/figures/feature_importance')
fig_imp.show()

## 5. Genre Distribution by Segment

Two complementary views:
- **Top genres**: proportion of plays per genre, stacked by cluster
- **Genre feature breakdown**: per-cluster ratios of unique genres, entropy, concentration, and avg tags (normalised to global mean)

In [ ]:
from src.visualization.plots import plot_genre_distribution, plot_genre_features, save_figure

# --- 5a. Stacked bar: top genre proportions per cluster ---
# Requires artist_genres.parquet (run notebook 01 Spotify enrichment cells to generate it)
if artist_genres is not None:
    fig_genre = plot_genre_distribution(
        scrobbles=scrobbles,
        artist_genres=artist_genres,
        labels=labels,
        userids=userids,
        top_n_genres=12,
        cluster_names=cluster_names,
    )
    save_figure(fig_genre, '../outputs/figures/genre_distribution')
    fig_genre.show()
else:
    print('Skipping genre distribution chart — artist_genres.parquet missing.')

# --- 5b. Per-cluster genre feature breakdown ---
# Uses pre-computed columns in user_features.parquet — no artist_genres needed
fig_genre_feats = plot_genre_features(
    feature_matrix=feature_matrix,
    labels=labels,
    cluster_names=cluster_names,
)
save_figure(fig_genre_feats, '../outputs/figures/genre_features')
fig_genre_feats.show()


## 6. Temporal Listening Patterns (per segment)

Three complementary views per segment:
- **Ratio breakdown**: morning (06–12h), evening (18–00h), and weekend fractions side-by-side
- **Hour-of-day distribution**: normalised listening curves across all 24 hours
- **Activity heatmap**: hour × day-of-week density (one heatmap per cluster)

In [ ]:
from src.visualization.plots import (
    plot_temporal_heatmap,
    plot_temporal_ratios,
    plot_hour_distribution,
    save_figure,
)

# --- 6a. Morning / Evening / Weekend ratio breakdown per cluster ---
fig_ratios = plot_temporal_ratios(
    feature_matrix=feature_matrix,
    labels=labels,
    cluster_names=cluster_names,
)
save_figure(fig_ratios, '../outputs/figures/temporal_ratios')
fig_ratios.show()

# --- 6b. Normalised hour-of-day distribution per cluster ---
fig_hours = plot_hour_distribution(
    scrobbles=scrobbles,
    labels=labels,
    userids=userids,
    cluster_names=cluster_names,
)
save_figure(fig_hours, '../outputs/figures/temporal_hour_distribution')
fig_hours.show()

# --- 6c. Hour x day-of-week activity heatmap (one per cluster) ---
for cid in sorted(set(labels)):
    if cid == -1:
        continue
    fig_temp = plot_temporal_heatmap(
        scrobbles=scrobbles,
        labels=labels,
        userids=userids,
        cluster_id=cid,
        cluster_name=cluster_names.get(cid, f'Cluster {cid}'),
    )
    save_figure(fig_temp, f'../outputs/figures/temporal_heatmap_cluster_{cid}')
    fig_temp.show()


## 6.5 Audio Feature Profile by Segment

Mean Spotify audio features per listener segment. Valence, energy, danceability, acousticness, instrumentalness, and speechiness are 0–1 normalised. Tempo is in BPM; loudness is in dB (typically −60 to 0).

In [ ]:
_AUDIO_DISPLAY = {
    'mean_valence':           'Valence (0–1)',
    'mean_energy':            'Energy (0–1)',
    'mean_danceability':      'Danceability (0–1)',
    'mean_acousticness':      'Acousticness (0–1)',
    'mean_instrumentalness':  'Instrumentalness (0–1)',
    'mean_tempo':             'Tempo (BPM)',
    'mean_loudness':          'Loudness (dB)',
    'mean_speechiness':       'Speechiness (0–1)',
}

fm_audio = feature_matrix.copy()

# mean_loudness is missing from parquets built before the engagement.py fix;
# compute it on-the-fly from audio_features.parquet if needed
if 'mean_loudness' not in fm_audio.columns:
    _af_path = Path('../data/processed/audio_features.parquet')
    if _af_path.exists():
        _af = pd.read_parquet(_af_path)
        if 'loudness' in _af.columns:
            _sc = scrobbles.copy()
            _sc['track_key'] = (_sc['artist_name'].str.strip().str.lower()
                                + '|||'
                                + _sc['track_name'].str.strip().str.lower())
            _merged = _sc.merge(
                _af[['track_key', 'loudness']].dropna(subset=['loudness']),
                on='track_key', how='left'
            )
            fm_audio = fm_audio.join(
                _merged.groupby('userid')['loudness'].mean().rename('mean_loudness'),
                how='left'
            )

available_cols = [c for c in _AUDIO_DISPLAY if c in fm_audio.columns]
fm_audio['_segment'] = pd.Series(labels, index=feature_matrix.index).map(cluster_names)

audio_table = (
    fm_audio[fm_audio['_segment'].notna()]
    .groupby('_segment')[available_cols]
    .mean()
    .rename(columns={k: v for k, v in _AUDIO_DISPLAY.items() if k in available_cols})
)

# Round: BPM to 1 dp, dB to 2 dp, all 0-1 features to 3 dp
for col in audio_table.columns:
    if 'BPM' in col:
        audio_table[col] = audio_table[col].round(1)
    elif 'dB' in col:
        audio_table[col] = audio_table[col].round(2)
    else:
        audio_table[col] = audio_table[col].round(3)

audio_table.index.name = 'Segment'
display(audio_table.T)  # transposed: features as rows, segments as columns
print()
print(audio_table.T.to_string())  # plain-text version for copy-paste


## 7. Business Insights & Personalization Recommendations

The cell below prints a structured summary of each segment's characteristics and actionable implications for streaming platforms.

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters

cluster_summary = summarise_clusters(feature_matrix, labels)
cluster_names_auto = label_clusters(cluster_summary, feature_matrix, labels)

# Key feature means per cluster (unscaled)
fm = feature_matrix.copy()
fm['cluster'] = labels

insight_features = [
    'artist_entropy', 'genre_entropy', 'unique_genres',
    'genre_concentration_5', 'avg_genre_tags_per_play',
    'unique_artists', 'track_replay_rate', 'novelty_ratio',
    'discovery_velocity_30d', 'avg_tracks_per_session',
    'weekend_ratio', 'temporal_hour_entropy',
]
insight_features = [f for f in insight_features if f in fm.columns]

global_means = fm[insight_features].mean()

print('=' * 72)
print('LISTENER SEGMENT PROFILES & PERSONALIZATION RECOMMENDATIONS')
print('=' * 72)

for cid in sorted(set(labels)):
    if cid == -1:
        continue
    cluster_users = fm[fm['cluster'] == cid]
    n = len(cluster_users)
    name = cluster_names.get(cid, f'Cluster {cid}')
    means = cluster_users[insight_features].mean()

    print(f'\nSEGMENT {cid}: {name.upper()}  ({n} users, {n/len(fm)*100:.1f}%)')
    print('-' * 60)

    for feat in insight_features:
        val = means[feat]
        gval = global_means[feat]
        delta = (val - gval) / (gval + 1e-9)
        arrow = '▲' if delta > 0.15 else ('▼' if delta < -0.15 else '—')
        print(f'  {feat:<35} {val:8.3f}  {arrow} (global: {gval:.3f})')

    # Heuristic recommendations
    recs = []
    if means.get('novelty_ratio', 0) > global_means.get('novelty_ratio', 0):
        recs.append('→ Prioritise new artist recommendations and discovery playlists')
    else:
        recs.append('→ Emphasise "More like your favourites" and artist radio')

    if means.get('track_replay_rate', 0) > global_means.get('track_replay_rate', 0) * 1.2:
        recs.append('→ Offer offline mode / download prompts for favourite tracks')

    if means.get('genre_entropy', 0) > global_means.get('genre_entropy', 0):
        recs.append('→ Cross-genre mood playlists and genre-blend features work well')
    else:
        recs.append('→ Deep-dive genre playlists and artist discography features')

    if means.get('unique_genres', 0) < global_means.get('unique_genres', 1) * 0.7:
        recs.append('→ Genre familiarity badge / stats to reinforce niche identity')

    if means.get('genre_concentration_5', 0) > 0.8:
        recs.append('→ Highly concentrated genre taste — lean into artist radio and discography deep-dives')

    if means.get('avg_genre_tags_per_play', 0) > global_means.get('avg_genre_tags_per_play', 1) * 1.2:
        recs.append('→ Genre-blend / fusion playlist recommendations resonate with this segment')

    if means.get('weekend_ratio', 0) > 0.45:
        recs.append('→ Target weekend push notifications and curated weekend playlists')

    if means.get('avg_tracks_per_session', 0) > global_means.get('avg_tracks_per_session', 0) * 1.3:
        recs.append('→ Long-session features: auto-queuing, seamless transitions, sleep timer')

    print('\n  PLATFORM RECOMMENDATIONS:')
    for r in recs:
        print(f'  {r}')

print('\n' + '=' * 72)

## 8. Demographic Cross-Tabulation (Descriptive Only)

In [ ]:
demo = profiles.set_index('userid')[['gender', 'age', 'country']]
labeled = labels_df.set_index('userid').join(demo)

print('Gender distribution per segment:')
print(labeled.groupby('cluster_name')['gender'].value_counts(normalize=True).round(3))

print('\nMedian age per segment:')
print(labeled.groupby('cluster_name')['age'].median())

print('\nTop 3 countries per segment:')
print(
    labeled.groupby('cluster_name')['country']
    .value_counts()
    .groupby(level=0)
    .head(3)
)